# Notebook 2 [KAGGLE] — IMX500 Export Pipeline [Phases 3–5]
**Input:** `best.pt` uploaded as Kaggle Dataset input  
**Output:** `packerOut.zip` saved to `/kaggle/working/` → Output tab  
**Runtime:** GPU T4 x2 (preferred) or P100

> 📄 [Sony IMX500 Export — Ultralytics Docs](https://docs.ultralytics.com/integrations/sony-imx500/)  
> 📄 [Sony MCT](https://github.com/SonySemiconductorSolutions/mct-model-optimization)  
> 📄 [aitrios-rpi deployment](https://github.com/SonySemiconductorSolutions/aitrios-rpi-application-module-library)

---
## Why Kaggle over Colab for NB2?
```
Phase 4 (IMX500 Java compiler) peak RAM: ~14–16 GB
Colab free T4 RAM : ~12 GB  → OOM crash
Kaggle free RAM   : ~16 GB  → sufficient headroom
```

---
## Phases covered
```
Phase 3 — MCT Quantization : FP32 best.pt → INT8 (PTQ on drone images)
Phase 4 — IMX500 Compile   : INT8 ONNX → IMX500 binary (Java, high RAM)
Phase 5 — Package          : binary → packerOut.zip

All 3 phases run inside model.export(format='imx') in B3.
```

---
## ⚠ Known Issues & Fixes Applied
| Issue | Fix Applied |
|---|---|
| `ultralytics[export]` no longer pulls MCT | Explicit pinned install in B1.2 |
| Mid-run package swaps crash B3 | Pre-pin cell added before B3 |
| Java OOM kills kernel silently | `JAVA_TOOL_OPTIONS=-Xmx8g` (Kaggle has more RAM) |

---
## How to set up and run this notebook on Kaggle

### Step 1 — Upload this notebook
1. Go to [kaggle.com](https://www.kaggle.com) → **Create** → **New Notebook**
2. In the new notebook: **File** → **Import Notebook** → upload `NB2_kaggle_yolo11n_drone_imx500_export.ipynb`

### Step 2 — Set runtime to GPU
1. Right panel → **Session Options** (or Settings icon)
2. **Accelerator** → select **GPU T4 x2** (preferred) or **P100**
3. **Persistence** → leave as default

### Step 3 — Enable Internet access
1. Right panel → **Session Options**
2. Toggle **Internet** → **On**  
   ⚠ Without this, pip installs and HuggingFace downloads fail silently

### Step 4 — Upload best.pt as a Kaggle Dataset
1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
2. Upload your `best.pt` file from NB1
3. Name it: **`drone-best-pt`**
4. Set visibility: **Private**
5. Click **Create**
6. Back in this notebook → right panel → **Add Input** → search **drone-best-pt** → **Add**
7. Kaggle mounts it at `/kaggle/input/drone-best-pt/best.pt` automatically

### Step 5 — Add HF_TOKEN secret (optional)
1. Top menu → **Add-ons** → **Secrets**
2. Click **Add a new secret** → Name: `HF_TOKEN`, Value: your token
3. Toggle it **On** for this notebook

### Step 6 — Run
1. **Run All** (or run cells in order B0 → B4)
2. B1.2 will trigger a runtime restart on first run — expected
3. After restart: run B0 → B1.1 → B1.2 (skips) → B1.3 → B2 → B3-pre-pin → B3 → B4

### Step 7 — Download output
1. Right panel → **Output** tab
2. Find `drone_imx_model.zip` → click to download
3. Unzip → transfer `packerOut.zip` to RPi5

### Step 8 — If B3 crashes
1. **Stop session** → **Start session** again (equivalent to Factory Reset)
2. Re-run B0 → B1.1 → B1.2 (skips) → B1.3 → B2 → B3-pre-pin → B3
3. `best.pt` is loaded from Kaggle Dataset input — never lost

---
## B0 — Configuration

In [ ]:
import glob

# Auto-detect best.pt wherever Kaggle placed it
matches = glob.glob('/kaggle/input/**/best.pt', recursive=True)

if not matches:
    raise FileNotFoundError(
        'best.pt not found in /kaggle/input\n'
        '  → Add Input (right panel) → search drone-best-pt → Add'
    )

BEST_PT_INPUT = matches[0]
print(f'✓ Found best.pt at: {BEST_PT_INPUT}')

In [ ]:
print(BEST_PT_INPUT)

In [ ]:
# B0 — Config
# best.pt is loaded from Kaggle Dataset input (never uploaded manually)
# Everything else written to /kaggle/working/

BEST_PT       = '/kaggle/working/best.pt'              # ← working copy
DATASET_DIR   = '/kaggle/working/drone_dataset'
YAML_PATH     = '/kaggle/working/data.yaml'

import os, sys
print('✓ Config loaded')
print(f'  Python          : {sys.version.split()[0]}')
print(f'  best.pt source  : {BEST_PT_INPUT}')
print(f'  Working dir     : /kaggle/working/')

---
## B1 — Dependency Install + Java 17

**B1.2 will trigger a runtime restart on first run — expected.**  
After restart: re-run B0 → B1.1 → B1.2 (skips) → B1.3 → continue from B2.

In [ ]:
# # B1.1 — Install Java 17
import subprocess, os

# Check if Java 17 is already present (survives restarts on persistent sessions)
probe = subprocess.run(['java', '-version'], capture_output=True, text=True)
if 'openjdk version "17' in probe.stderr:
    print('✓ Java 17 already installed — skipping apt-get')
else:
    print('Installing Java 17...')
    subprocess.run(['apt-get', 'update', '-q'], check=True)                           # ← add this
    subprocess.run(['apt-get', 'install', '-y', '-q', '--fix-missing',                # ← add --fix-missing
                    'openjdk-17-jdk'], check=True)

os.environ['JAVA_HOME']        = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH']             = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
os.environ['JAVA_TOOL_OPTIONS'] = '-Xmx10g'

result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print('✓ Java:', result.stderr.strip().split('\n')[0])


In [ ]:
# B1.2 — Install all IMX500 dependencies (pinned versions)
#
# WHY PINNED VERSIONS:
#   Ultralytics' export(format='imx') expects specific versions of MCT,
#   imx500-converter, onnx, and protobuf. Without pinning, Ultralytics
#   performs live package swaps mid-B3 → runtime crash.
#
# WHY IMPORT GUARD:
#   After restart, pip packages persist but /kaggle/working/ is wiped.
#   If MCT imports → already installed → skip → no second restart.

try:
    import model_compression_toolkit as mct
    import ultralytics
    print('✓ Dependencies already installed — no restart needed')
    print(f'  MCT        : {mct.__version__}')
    print(f'  Ultralytics: {ultralytics.__version__}')

except ImportError:
    print('Installing IMX500 export dependencies (pinned)...')
    print('Runtime restart will follow — expected.\n')

    import subprocess
    subprocess.run(['pip', 'install', 'ultralytics', 'huggingface_hub', '--quiet'], check=True)
    subprocess.run(['pip', 'install',
                    'model-compression-toolkit==2.4.5',
                    'edge-mdt-cl==1.0.0',
                    'edge-mdt-tpc>=1.2.0',
                    'imx500-converter[pt]>=3.17.3',
                    'pydantic<=2.11.7',
                    '--quiet'], check=True)

    # Conflict warnings about tensorflow/grpcio/google-cloud-* are safe to ignore.
    # These are Kaggle pre-installed packages complaining about protobuf downgrade.
    # None are used by the IMX500 export pipeline.

    print('\n✓ Install complete — restarting runtime...')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# B1.3 — Verify all imports
import torch
import model_compression_toolkit as mct
from ultralytics import YOLO

if torch.cuda.is_available():
    print(f'✓ GPU     : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('✗ No GPU — right panel → Session Options → Accelerator → GPU T4 x2')

print(f'✓ Python  : {__import__("sys").version.split()[0]}')
print(f'✓ MCT     : {mct.__version__}')
print(f'✓ YOLO    : {__import__("ultralytics").__version__}')
print('\n→ All checks passed — proceed to B2')

---
## B2 — Load best.pt + Dataset + data.yaml
**best.pt is read from Kaggle Dataset input — safe across all restarts.**

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# B2.1 — Copy best.pt from Kaggle Dataset input to working directory
import shutil, os

if not os.path.exists(BEST_PT_INPUT):
    raise FileNotFoundError(
        f'best.pt not found at {BEST_PT_INPUT}\n'
        f'  → Go to Add Input (right panel) → search drone-best-pt → Add\n'
        f'  → Or re-upload best.pt as a new Kaggle Dataset named drone-best-pt'
    )

shutil.copy(BEST_PT_INPUT, BEST_PT)
print(f'✓ best.pt loaded from Kaggle Dataset input')
print(f'  Size: {os.path.getsize(BEST_PT)/1e6:.1f} MB')

In [ ]:
# B2.2 — Download drone dataset (for MCT calibration)
# fraction=0.10 is safe on Kaggle (more RAM than Colab)
from huggingface_hub import snapshot_download
import os

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    print('✓ HF_TOKEN loaded from Kaggle Secrets')
except Exception:
    hf_token = os.environ.get('HF_TOKEN', None)
    print('ℹ HF_TOKEN not found — using unauthenticated (works fine)')

print('Downloading drone dataset for MCT calibration...')
snapshot_download(
    repo_id  = 'lgrzybowski/seraphim-drone-detection-dataset',
    repo_type= 'dataset',
    local_dir= DATASET_DIR,
    token    = hf_token
)
print('\n✓ Dataset downloaded')

In [ ]:
# B2.3 — Extract zip batches
import zipfile, glob
zip_files = glob.glob(os.path.join(DATASET_DIR, '**', '*.zip'), recursive=True)
print(f'Found {len(zip_files)} zip files. Extracting...')
for i, zp in enumerate(sorted(zip_files)):
    with zipfile.ZipFile(zp, 'r') as zf:
        zf.extractall(os.path.dirname(zp))
    os.remove(zp)
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(zip_files)}...')
print('\n✓ Extraction complete')

In [ ]:
# B2.4 — Write data.yaml
import yaml
with open(YAML_PATH, 'w') as f:
    yaml.dump({
        'path' : DATASET_DIR,
        'train': 'train/images',
        'val'  : 'test/images',
        'nc'   : 1,
        'names': ['drone']
    }, f, default_flow_style=False)
print('✓ data.yaml written')
with open(YAML_PATH) as f: print(f.read())

---
## B3-pre-pin — Force correct package versions before export
**Run immediately before B3 every time.**

In [ ]:
# B3-pre-pin — Pin all IMX500 export dependencies
#
# WHY THIS IS NEEDED EVEN AFTER B1.2:
#   Ultralytics checks its expected versions at export time and may attempt
#   live re-installs mid-run if anything drifted. Running this right before
#   B3 locks everything into the correct state.
#
# ABOUT CONFLICT WARNINGS:
#   ERROR messages about tensorflow, grpcio, google-cloud-* are safe to ignore.
#   Kaggle's own pre-installed packages complaining about protobuf downgrade.
#   None used by the IMX500 export pipeline.

import subprocess
subprocess.run(['pip', 'install',
                'edge-mdt-cl==1.0.0',
                'edge-mdt-tpc>=1.2.0',
                'imx500-converter[pt]==3.17.3',
                'model-compression-toolkit==2.4.5',
                'pydantic<=2.11.7',
                '--quiet'], check=True)

# Cap Java heap — Kaggle has ~16GB so 8GB cap gives headroom without OOM
# import os
# os.environ['JAVA_TOOL_OPTIONS'] = '-Xmx8g'  # ← 8GB on Kaggle vs 6GB on Colab

import os
os.environ['JAVA_TOOL_OPTIONS'] = (
    '-Xmx6g '
    '-XX:MaxDirectMemorySize=4g '
    '-XX:+UseG1GC '
    '-XX:MaxHeapFreeRatio=30'
)

print('✓ IMX500 deps pre-pinned')
print('✓ Java heap capped at 8GB')
print('→ Proceed to B3')

In [ ]:
# Pre-B3: re-assert clean JVM flags (survives kernel restarts)
import os
os.environ['JAVA_TOOL_OPTIONS'] = '-Xmx10g'
print('✓ JAVA_TOOL_OPTIONS:', os.environ['JAVA_TOOL_OPTIONS'])

# Verify no conflicting flags survived
import subprocess
result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print('✓ Java:', result.stderr.strip().split('\n')[0])

---
## B3 — Export to IMX500 [Phases 3 + 4 + 5]
```
Phase 3 — MCT PTQ   : samples fraction of dataset → FP32 → INT8
Phase 4 — Converter : INT8 ONNX → IMX500 binary (Java, ~10–20 min)
Phase 5 — Packager  : binary → packerOut.zip
```
> 📄 [Export arguments](https://docs.ultralytics.com/integrations/sony-imx500/#export-arguments)

In [ ]:
# B3 — Export to IMX500
# Reference: https://docs.ultralytics.com/integrations/sony-imx500/
from ultralytics import YOLO

model = YOLO(BEST_PT)

# ─────────────────────────────────────────────────────────────────
# fraction: portion of training dataset used by MCT for PTQ calibration.
#   Controls Phase 3 only — Phase 4 Java compilation is unaffected.
# ✓  test   images= 8,349  labels= 8,349
#   0.05 → ~417 images — minimum useful (Colab)
#   0.10 → ~834 images — RECOMMENDED on Kaggle (safe with 16GB RAM)
#   0.25 → ~2087 images — thorough; Kaggle only
# ─────────────────────────────────────────────────────────────────

model.export(
    format   = 'imx',
    data     = YAML_PATH,
    imgsz    = 320,    # changed from 640, inorder to test for inference improvement
    int8     = True,
    fraction = 1,   # ← 0.10 on Kaggle (more RAM than Colab's 0.05)
    device   = 0
)

import os, glob
imx_dirs = (glob.glob('/kaggle/working/**/*imx_model', recursive=True) +
            glob.glob('/kaggle/working/*imx_model'))

if imx_dirs:
    imx_dir = imx_dirs[0]
    print(f'\n✓ Export complete: {imx_dir}')
    for f in sorted(os.listdir(imx_dir)):
        size = os.path.getsize(os.path.join(imx_dir, f)) / 1024
        print(f'  {f:45s}  {size:8.1f} KB')
    print('\n  → packerOut.zip is your RPi5 deployment artifact')
else:
    print('⚠ IMX folder not found — listing /kaggle/working:')
    for item in os.listdir('/kaggle/working'): print(f'  {item}')

---
## B4 — Save outputs to Output tab

In [ ]:
# B4 — Zip and save final outputs to /kaggle/working/ → Output tab
# No files.download() needed — Kaggle Output tab shows all /kaggle/working/ files.
import shutil, os, glob

imx_dirs = (glob.glob('/kaggle/working/**/*imx_model', recursive=True) +
            glob.glob('/kaggle/working/*imx_model'))

if imx_dirs:
    zip_out = '/kaggle/working/drone_imx_model'
    shutil.make_archive(zip_out, 'zip', imx_dirs[0])
    print('✓ drone_imx_model.zip → Output tab (right panel)')
    print('  → Download from Output tab')
    print('  → Unzip → transfer packerOut.zip to RPi5')
    print('  → Guide: https://docs.ultralytics.com/integrations/sony-imx500/#using-imx500-export-in-deployment')
else:
    print('⚠ IMX model directory not found — ensure B3 completed successfully')

---
## Package Version Reference
| Package | Version |
|---|---|
| `model-compression-toolkit` | 2.4.5 |
| `mct-quantizers` | 1.6.0 |
| `edge-mdt-cl` | 1.0.0 |
| `edge-mdt-tpc` | 1.2.0 |
| `imx500-converter` | 3.18.2 |
| `pydantic` | ≤ 2.11.7 |
| `onnx` | 1.17.0 |
| `protobuf` | 4.25.5 |

---
## Key References
| Topic | Link |
|---|---|
| Sony IMX500 Export | https://docs.ultralytics.com/integrations/sony-imx500/ |
| Export arguments | https://docs.ultralytics.com/integrations/sony-imx500/#export-arguments |
| RPi5 deployment | https://docs.ultralytics.com/integrations/sony-imx500/#using-imx500-export-in-deployment |
| Sony MCT | https://github.com/SonySemiconductorSolutions/mct-model-optimization |
| aitrios-rpi | https://github.com/SonySemiconductorSolutions/aitrios-rpi-application-module-library |

In [ ]:
from ultralytics import YOLO
model = YOLO(BEST_PT)
metrics = model.val(data=YAML_PATH, imgsz=640, device=0)

print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST_PT)
print(model.info())          # parameter count, layer count
print(model.model.yaml)      # full architecture dict